In [49]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 

from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Dense 
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import re

In [50]:
df=pd.read_csv('IMDB Dataset.csv')

In [51]:
print(df.isnull().sum())

review       0
sentiment    0
dtype: int64


In [52]:
df.columns=['review','sentiment']

In [53]:
def clean_text(text):
    text=str(text)
    text=text.lower()
    text=re.sub(r'<.*?>','',text)
    text=re.sub('[^a-zA-Z\s]','',text)
    return text
df['review']=df['review'].apply(clean_text)
    

<>:5: SyntaxWarning: invalid escape sequence '\s'
<>:5: SyntaxWarning: invalid escape sequence '\s'
C:\Users\HP\AppData\Local\Temp\ipykernel_16728\3149390700.py:5: SyntaxWarning: invalid escape sequence '\s'
  text=re.sub('[^a-zA-Z\s]','',text)


In [54]:
encoder=LabelEncoder()
df['sentiment'] = encoder.fit_transform(df['sentiment'])
print(df['sentiment'].value_counts())

sentiment
1    25000
0    25000
Name: count, dtype: int64


In [55]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer=TfidfVectorizer(max_features=5000)
X=vectorizer.fit_transform(df['review'])
y=df['sentiment']

In [56]:
y = df['sentiment'].reset_index(drop=True)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)


In [57]:
model=Sequential()
model.add(Dense(128,activation='relu',input_shape=(X_train.shape[1],)))
model.add(Dense(64,activation='relu'))
model.add(Dense(1,activation='sigmoid'))

model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
model.summary()

C:\Users\HP\anaconda4\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                      │ (None, 128)                 │         640,128 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_7 (Dense)                      │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_8 (Dense)                      │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 648,449 (2.47 MB)

 Trainable params: 648,449 (2.47 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=32)

Epoch 1/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 17s 9ms/step - accuracy: 0.8692 - loss: 0.3084
Epoch 2/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 11s 8ms/step - accuracy: 0.9059 - loss: 0.2313
Epoch 3/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 9s 7ms/step - accuracy: 0.9292 - loss: 0.1808
Epoch 4/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - accuracy: 0.9701 - loss: 0.0859
Epoch 5/5
 906/1250 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9943 - loss: 0.0228

In [ ]:
loss,accuracy=model.evaluate(X_train,y_train)
print("accuracy",accuracy)

In [ ]:
y_pred=model.predict(X_test)
y_pred=(y_pred > 0.5).astype(int)
y_pred=y_pred.flatten()

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:

import seaborn as sns
cm=confusion_matrix(y_test,y_pred)
plt.figure(figsize=(10,6))
sns.heatmap(cm,annot=True,fmt='d',xticklabels=['negative','positive'],yticklabels=['negative','positive'])


plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')

plt.show()

print(classification_report(y_test,y_pred))